In [1]:
!pip install litellm==1.81.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.9 MB/s eta 0:00:00


In [4]:
# set openrouter API key
import os
os.environ["OPENROUTER_API_KEY"] = "..."

Sign up and obtain your API key from [https://openrouter.ai/](https://openrouter.ai/).

By default, we use **gpt-4o-mini**. However, OpenRouter also provides several free models that you can access using the same API key.

For example:
`openrouter/meta-llama/llama-3.3-70b-instruct:free`, `openrouter/meta-llama/llama-3.1-405b-instruct:free`

**Alternative option**

You can also use **Groq** instead of OpenRouter. Groq offers limited free inference for certain models as well.
With LiteLLM, the usage format is:

`groq/<model-name>`

You can refer to the official LiteLLM Groq provider guide here:
[https://docs.litellm.ai/docs/providers/groq](https://docs.litellm.ai/docs/providers/groq)

**Final note**

While free models are supported, we recommend using **gpt-4o-mini**. Free models are often under heavy demand, which can lead to frequent rate limits or API failures. A small top-up on OpenRouter is usually worth it.

Or, if you have an OpenAI key, then you can switch providers entirely and use OpenAI directly with LiteLLM, for example: `openai/gpt-4o-mini` instead of `openrouter/openai/gpt-4o-mini`, and use your OpenAI API key instead of OpenRouter key.

In [ ]:
import re
from datetime import datetime

import litellm
litellm.suppress_debug_info = True
litellm.set_verbose = False

from litellm import completion

MODEL = "openrouter/openai/gpt-4o-mini"

system_prompt = """
You are an expert mathematics tutor and problem solver.

You must produce TWO parts in this exact order:
1) A step-by-step solution inside <thought>...</thought> tags showing all your reasoning.
2) The final solution in STRICT Markdown format exactly as specified.

Do not add anything else outside these two parts.
"""

SOLUTION_FORMAT_SPEC = """
## SOLUTION_START

### PROBLEM_UNDERSTANDING
<Brief restatement of what the problem is asking>

### APPROACH
<The method or strategy used to solve the problem>

### SOLUTION_STEPS
1. <Step 1>
2. <Step 2>
3. <Step 3>
...

### FINAL_ANSWER
<The final numerical answer with appropriate units>

### VERIFICATION
<Quick check that the answer makes sense>

## SOLUTION_END
""".strip()


def extract_and_strip_thoughts(text: str) -> tuple[str, str]:
    """
    Extracts ALL <thought>...</thought> blocks and removes them from the text.
    Returns (cleaned_text, combined_thoughts).
    """
    pattern = re.compile(r"<thought>\s*(.*?)\s*</thought>", re.DOTALL | re.IGNORECASE)

    # Find all thought blocks
    matches = pattern.findall(text)

    # Combine all thoughts with separators
    if matches:
        thought = "\n\n--- THOUGHT BLOCK ---\n\n".join(m.strip() for m in matches)
    else:
        thought = ""

    # Remove all thought blocks from the text
    cleaned = pattern.sub("", text).strip()

    # Remove excessive blank lines
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip()

    return cleaned, thought


def save_thought_to_file(thought: str, filename: str = "math_cot_reasoning.txt") -> None:
    """
    Appends the extracted chain-of-thought reasoning to a txt file.
    """
    if not thought:
        return

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"\n{'='*60}\n[{timestamp}]\n{thought}\n"
    with open(filename, "a", encoding="utf-8") as f:
        f.write(entry)


def solve_math_problem(problem: str) -> str:
    """
    Solves a math word problem using chain-of-thought reasoning.

    Args:
        problem: The math word problem text

    Returns:
        Formatted solution (without the reasoning)
    """
    user_prompt = f"""
Solve the following math word problem.

Your tasks:
1) Show your complete step-by-step reasoning inside <thought>...</thought> tags.
   - Identify what's given and what's being asked
   - Plan your approach
   - Work through each calculation with explanations
   - Check your work
2) Output the final solution in the STRICT Markdown format below.

STRICT Markdown format (must match exactly):

{SOLUTION_FORMAT_SPEC}

Math problem to solve:
\"\"\"
{problem}
\"\"\"
""".strip()

    resp = completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    raw = resp.choices[0].message.content

    # Extract and filter out ALL thought blocks
    final_output, chain_of_thought = extract_and_strip_thoughts(raw)

    # Save the reasoning to file
    save_thought_to_file(chain_of_thought, filename="math_cot_reasoning.txt")

    return final_output


if __name__ == "__main__":
    sample_problem = """
Sarah has 3 boxes of cookies. Each box contains 24 cookies.
She wants to distribute all the cookies equally among 8 friends.
How many cookies will each friend receive? Will there be any cookies left over?
""".strip()

    solution = solve_math_problem(sample_problem)
    print(solution)
    print("\n(Chain-of-thought reasoning saved to ./math_cot_reasoning.txt)")

## SOLUTION_START

### PROBLEM_UNDERSTANDING
Sarah has 3 boxes of cookies, each containing 24 cookies, and wants to distribute them equally among 8 friends. We need to determine how many cookies each friend receives and if there are any leftovers.

### APPROACH
Calculate the total number of cookies, divide that total by the number of friends to find out how many cookies each friend gets, and check for any remainder to see if there are leftovers.

### SOLUTION_STEPS
1. Calculate the total number of cookies: 3 boxes × 24 cookies/box = 72 cookies.
2. Divide the total cookies by the number of friends: 72 cookies ÷ 8 friends = 9 cookies per friend.
3. Check for leftovers: 72 cookies % 8 friends = 0 cookies left over.

### FINAL_ANSWER
Each friend will receive 9 cookies, and there will be no cookies left over.

### VERIFICATION
The calculations confirm that 72 cookies divided by 8 friends results in 9 cookies each, with no remainder, which makes sense given the total number of cookies.

## S